In [1]:
# ============================================================
# UHTM Dataset Generator — 200 samples × 40 features
# ML-Based Multi-Objective Optimization of Ultra-High
# Temperature Materials for Aerospace Applications
# ============================================================

import pandas as pd
import numpy as np

np.random.seed(42)

# ── Material Base Properties ─────────────────────────────────
MATERIAL_CLASSES = {
    "HfC":  {"melting_point": 3900, "density": 12.2,  "crystal": "FCC", "valence_e": 8,  "electronegativity_diff": 1.3},
    "ZrC":  {"melting_point": 3420, "density": 6.73,  "crystal": "FCC", "valence_e": 8,  "electronegativity_diff": 1.3},
    "TaC":  {"melting_point": 3880, "density": 14.3,  "crystal": "FCC", "valence_e": 9,  "electronegativity_diff": 1.1},
    "HfB2": {"melting_point": 3380, "density": 10.5,  "crystal": "HEX", "valence_e": 6,  "electronegativity_diff": 0.9},
    "ZrB2": {"melting_point": 3245, "density": 6.09,  "crystal": "HEX", "valence_e": 6,  "electronegativity_diff": 0.9},
    "TiC":  {"melting_point": 3160, "density": 4.93,  "crystal": "FCC", "valence_e": 8,  "electronegativity_diff": 1.5},
    "NbC":  {"melting_point": 3600, "density": 7.79,  "crystal": "FCC", "valence_e": 9,  "electronegativity_diff": 1.2},
    "HfN":  {"melting_point": 3385, "density": 13.8,  "crystal": "FCC", "valence_e": 9,  "electronegativity_diff": 1.6},
    "ZrN":  {"melting_point": 2980, "density": 7.09,  "crystal": "FCC", "valence_e": 9,  "electronegativity_diff": 1.6},
    "TaN":  {"melting_point": 3090, "density": 16.3,  "crystal": "HEX", "valence_e": 10, "electronegativity_diff": 1.4},
}

COMPOSITES = ["HfC-SiC","ZrB2-SiC","HfB2-SiC","TaC-HfC","ZrC-TiC",
              "HfC-TaC","ZrB2-ZrC","HfB2-MoSi2","TiC-TiB2","NbC-HfC"]

DOPED = ["HfC:Y","ZrC:La","TaC:W","HfB2:Al","ZrB2:Y",
         "TiC:Nb","NbC:Ta","HfN:Zr","ZrN:Hf","TaN:Nb"]

SYNTHESIS_METHODS = ["Spark Plasma Sintering","Hot Pressing","Arc Melting",
                     "Chemical Vapor Deposition","Reactive Sintering",
                     "Pressureless Sintering","Tape Casting + Sintering"]

def noise(val, pct=0.05):
    return val * (1 + np.random.normal(0, pct))

def gen_row(idx, source_type):
    is_composite = (source_type == "experimental") and (idx >= 50)
    is_doped     = (source_type == "synthetic")    and (idx >= 150)

    base_keys = list(MATERIAL_CLASSES.keys())
    bk = base_keys[idx % len(base_keys)]

    if is_composite:
        mat_name = COMPOSITES[(idx - 50) % len(COMPOSITES)]
    elif is_doped:
        mat_name = DOPED[(idx - 150) % len(DOPED)]
    else:
        mat_name = bk

    B   = MATERIAL_CLASSES[bk]
    mp  = B["melting_point"]
    rho = B["density"]
    ve  = B["valence_e"]
    en  = B["electronegativity_diff"]

    synth            = SYNTHESIS_METHODS[idx % len(SYNTHESIS_METHODS)]
    sintering_press  = noise(30 + ve * 3, 0.1)

    # ── Group A: Thermodynamic / Structural ──────────────────
    melting_point_K      = noise(mp, 0.02)
    debye_temp_K         = noise(300 + mp * 0.15, 0.04)
    cohesive_energy      = noise(ve * 0.82 + en * 1.1, 0.05)
    formation_enthalpy   = noise(-(en * 45 + ve * 8), 0.05)
    lattice_param_a      = noise(3.2 + (rho ** -0.3) * 0.5, 0.02)

    # ── Group B: Electronic Structure ────────────────────────
    band_gap_eV          = max(0, noise(0.5 - ve * 0.05, 0.2))
    DOS_at_Ef            = noise(ve * 0.55 + 1.2, 0.08)
    charge_transfer      = noise(en * 0.6, 0.07)
    Fermi_velocity       = noise(1e6 * (ve / 8.5), 0.06)
    electron_density     = noise(rho * ve * 1.8e22, 0.05)

    # ── Group C: Mechanical ───────────────────────────────────
    youngs_modulus       = noise(200 + mp * 0.04 + ve * 15, 0.05)
    hardness_vickers     = noise(15 + en * 8 + ve * 1.2, 0.06)
    fracture_toughness   = noise(2.5 + (1 / en) * 1.5, 0.07)
    compressive_str      = noise(youngs_modulus * 0.6, 0.05)
    poisson_ratio        = noise(0.18 + en * 0.02, 0.04)

    # ── Group D: Thermal Transport ────────────────────────────
    thermal_conductivity = noise(20 + ve * 4 - en * 3, 0.07)
    thermal_expansion    = noise(6.5 + en * 0.8 - ve * 0.3, 0.06)
    specific_heat        = noise(180 + (1 / rho) * 800, 0.05)
    thermal_diffusivity  = noise(thermal_conductivity / (rho * specific_heat / 1e3), 0.06)
    max_service_temp     = noise(mp * 0.75, 0.03)

    # ── Group E: Oxidation / Chemical Stability ───────────────
    oxidation_onset_T    = noise(800 + en * 150 + mp * 0.05, 0.05)
    oxidation_rate_const = noise(1e-12 * np.exp(en * 0.8), 0.10)
    activation_energy_ox = noise(120 + en * 30, 0.06)
    parabolic_rate_coef  = noise(1e-10 * en, 0.12)
    oxide_layer_stability= noise(en * 0.7 + ve * 0.15, 0.07)

    # ── Group F: Microstructural ──────────────────────────────
    grain_size_um        = noise(2 + (sintering_press ** -0.3) * 10, 0.10)
    relative_density     = min(99.9, noise(92 + sintering_press * 0.18, 0.02))
    porosity_pct         = max(0.1, noise(100 - relative_density, 0.15))
    crystallite_size_nm  = noise(grain_size_um * 1000 * 0.08, 0.12)
    dislocation_density  = noise(1e12 / (grain_size_um ** 1.5), 0.10)

    # ── Group G: Phase / Composite ────────────────────────────
    phase_stability_idx  = noise(mp / 4000 + cohesive_energy / 12, 0.06)
    secondary_phase_vol  = noise(5 + (idx % 30) * 0.5, 0.15) if is_composite else 0.0
    interfacial_energy   = noise(0.5 + en * 0.3, 0.08)
    CTE_mismatch         = noise(abs(thermal_expansion - 5.0) * 0.3, 0.10)
    solid_solution_delta = noise(en * 0.12 + (ve % 3) * 0.05, 0.08)

    # ── Group H: ML / Derived Descriptors ────────────────────
    merit_index_thermal  = thermal_conductivity / (thermal_expansion * rho)
    toughness_index      = fracture_toughness * youngs_modulus ** 0.5
    oxidation_merit      = oxidation_onset_T / (oxidation_rate_const * 1e12 + 1)
    bond_ionicity        = en / (en + ve * 0.15)
    structural_stability = (melting_point_K / 4000) * (cohesive_energy / 10) * (1 - porosity_pct / 100)

    # ── Targets ───────────────────────────────────────────────
    flexural_strength  = noise(300 + youngs_modulus * 0.8 + hardness_vickers * 12 - porosity_pct * 15, 0.05)
    ox_score           = min(10, max(0, noise(
                             3*(oxidation_onset_T/1500) + 2*(1/(oxidation_rate_const*1e10+0.1)) +
                             2*oxide_layer_stability + 1*(1-CTE_mismatch), 0.05)))
    thermal_shock      = max(1, int(noise(20 + fracture_toughness*8 + (1/thermal_expansion)*0.5 - CTE_mismatch*5, 0.08)))

    return {
        "Sample_ID":                        f"UHTM-{idx+1:03d}",
        "Material_System":                  mat_name,
        "Source_Type":                      source_type,
        "Synthesis_Method":                 synth,
        "Crystal_Structure":                B["crystal"],
        # Group A
        "F01_Melting_Point_K":              round(melting_point_K, 1),
        "F02_Debye_Temperature_K":          round(debye_temp_K, 1),
        "F03_Cohesive_Energy_eV":           round(cohesive_energy, 4),
        "F04_Formation_Enthalpy_kJmol":     round(formation_enthalpy, 3),
        "F05_Lattice_Param_a_Ang":          round(lattice_param_a, 4),
        # Group B
        "F06_Band_Gap_eV":                  round(band_gap_eV, 4),
        "F07_DOS_at_Fermi_statesEV":        round(DOS_at_Ef, 4),
        "F08_Charge_Transfer_e":            round(charge_transfer, 4),
        "F09_Fermi_Velocity_ms":            round(Fermi_velocity, 0),
        "F10_Valence_Electron_Density":     round(electron_density, 4),
        # Group C
        "F11_Youngs_Modulus_GPa":           round(youngs_modulus, 2),
        "F12_Vickers_Hardness_GPa":         round(hardness_vickers, 3),
        "F13_Fracture_Toughness_MPasm":     round(fracture_toughness, 4),
        "F14_Compressive_Strength_GPa":     round(compressive_str, 3),
        "F15_Poisson_Ratio":                round(poisson_ratio, 4),
        # Group D
        "F16_Thermal_Conductivity_WmK":     round(thermal_conductivity, 3),
        "F17_Thermal_Expansion_1e6K":       round(thermal_expansion, 4),
        "F18_Specific_Heat_JkgK":           round(specific_heat, 2),
        "F19_Thermal_Diffusivity_m2s":      round(thermal_diffusivity, 8),
        "F20_Max_Service_Temp_K":           round(max_service_temp, 1),
        # Group E
        "F21_Oxidation_Onset_Temp_K":       round(oxidation_onset_T, 1),
        "F22_Oxidation_Rate_Const_kg2m4s":  round(oxidation_rate_const, 20),
        "F23_Activation_Energy_Ox_kJmol":   round(activation_energy_ox, 3),
        "F24_Parabolic_Rate_Coef_g2cm4s":   round(parabolic_rate_coef, 16),
        "F25_Oxide_Layer_Stability_Idx":    round(oxide_layer_stability, 4),
        # Group F
        "F26_Grain_Size_um":                round(grain_size_um, 3),
        "F27_Relative_Density_pct":         round(relative_density, 3),
        "F28_Porosity_pct":                 round(porosity_pct, 4),
        "F29_Crystallite_Size_nm":          round(crystallite_size_nm, 2),
        "F30_Dislocation_Density_m2":       round(dislocation_density, 4),
        # Group G
        "F31_Phase_Stability_Index":        round(phase_stability_idx, 5),
        "F32_Secondary_Phase_vol_pct":      round(secondary_phase_vol, 2),
        "F33_Interfacial_Energy_Jm2":       round(interfacial_energy, 4),
        "F34_CTE_Mismatch":                 round(CTE_mismatch, 5),
        "F35_Solid_Solution_Delta":         round(solid_solution_delta, 5),
        # Group H
        "F36_Thermal_Merit_Index":          round(merit_index_thermal, 4),
        "F37_Toughness_Index":              round(toughness_index, 4),
        "F38_Oxidation_Merit_Score":        round(oxidation_merit, 4),
        "F39_Bond_Ionicity":                round(bond_ionicity, 5),
        "F40_Structural_Stability_Idx":     round(structural_stability, 6),
        # Targets
        "T1_Flexural_Strength_MPa":         round(flexural_strength, 1),
        "T2_Oxidation_Resistance_Score":    round(ox_score, 3),
        "T3_Thermal_Shock_Cycles":          thermal_shock,
    }

# ── Generate 200 rows ─────────────────────────────────────────
rows = []
for i in range(100):
    rows.append(gen_row(i, "experimental"))
for i in range(100):
    rows.append(gen_row(i + 100, "synthetic"))

df = pd.DataFrame(rows)

print(f"Dataset shape     : {df.shape}")
print(f"Experimental rows : {(df['Source_Type']=='experimental').sum()}")
print(f"Synthetic rows    : {(df['Source_Type']=='synthetic').sum()}")
print(f"Feature columns   : {len([c for c in df.columns if c.startswith('F')])}")
print(f"Target columns    : {len([c for c in df.columns if c.startswith('T')])}")
df.head()

Dataset shape     : (200, 48)
Experimental rows : 100
Synthetic rows    : 100
Feature columns   : 40
Target columns    : 3


,Sample_ID,Material_System,Source_Type,Synthesis_Method,Crystal_Structure,F01_Melting_Point_K,F02_Debye_Temperature_K,F03_Cohesive_Energy_eV,F04_Formation_Enthalpy_kJmol,F05_Lattice_Param_a_Ang,...,F34_CTE_Mismatch,F35_Solid_Solution_Delta,F36_Thermal_Merit_Index,F37_Toughness_Index,F38_Oxidation_Merit_Score,F39_Bond_Ionicity,F40_Structural_Stability_Idx,T1_Flexural_Strength_MPa,T2_Oxidation_Resistance_Score,T3_Thermal_Shock_Cycles
0,UHTM-001,HfC,experimental,Spark Plasma Sintering,FCC,3889.2,907.9,8.5985,-121.066,3.4200,...,0.06356,0.27285,0.6995,68.2324,305.7349,0.52000,0.835195,1029.6,10,38
1,UHTM-002,ZrC,experimental,Hot Pressing,FCC,3433.5,837.0,8.0585,-121.792,3.4612,...,0.14787,0.25527,1.4119,72.5616,331.0749,0.52000,0.691019,1142.8,10,50
2,UHTM-003,TaC,experimental,Arc Melting,FCC,3856.8,885.2,7.7363,-120.165,3.4496,...,0.12268,0.11174,0.7638,89.3415,344.1039,0.44898,0.745191,1097.7,10,61
3,UHTM-004,HfB2,experimental,Chemical Vapor Deposition,HEX,3400.4,805.9,5.5647,-93.557,3.4988,...,0.11517,0.11251,0.6443,86.9135,411.1106,0.50000,0.472576,1022.7,10,52
4,UHTM-005,ZrB2,experimental,Reactive Sintering,HEX,3260.1,796.0,5.6989,-96.756,3.5239,...,0.05621,0.11209,1.2880,86.0166,358.6843,0.50000,0.463963,990.0,10,47


In [ ]:
df.to_excel("UHTM_Dataset.xlsx", index=False)